In [1]:
from mysql import connector
from sqlalchemy import create_engine
import pandas as pd

## Load Data

In [2]:
df = pd.read_csv("../data/raw/Telco-Customer-Churn.csv")

## Inspect Data

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [4]:
df.isnull().sum()

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

In [5]:
df['customerID'].duplicated().sum()

np.int64(0)

## Converting Datatypes to match DB Schema

In [6]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

## Renaming the Columns to match DB Schema

In [7]:
df.rename(columns={'customerID': 'customer_id'}, inplace=True)

## Creating DataFrame for Each Table in DB

In [8]:
fact_customer_churn = df[['customer_id','MonthlyCharges','TotalCharges','Churn']].copy()

dim_customer = df[['customer_id','gender','SeniorCitizen','Partner','Dependents','tenure']].copy()

dim_services = df[['customer_id','PhoneService','MultipleLines','InternetService',
                   'OnlineSecurity','OnlineBackup','DeviceProtection',
                   'TechSupport','StreamingTV','StreamingMovies']].copy()

dim_account = df[['customer_id','Contract','PaperlessBilling','PaymentMethod']].copy()

## Insert into MySQL

In [9]:
def insert_df(df, table):
    cols = ",".join(df.columns)
    values = ",".join(["%s"] * len(df.columns))
    sql = f"INSERT INTO {table} ({cols}) VALUES ({values})"

    for row in df.itertuples(index=False):
        cursor.execute(sql, tuple(row))

    conn.commit()

In [10]:
engine = create_engine("mysql+pymysql://root:<your-password>@localhost/customer_db")

In [11]:
print(df.isnull().sum())
print(df.dtypes)

customer_id          0
gender               0
SeniorCitizen        0
Partner              0
Dependents           0
tenure               0
PhoneService         0
MultipleLines        0
InternetService      0
OnlineSecurity       0
OnlineBackup         0
DeviceProtection     0
TechSupport          0
StreamingTV          0
StreamingMovies      0
Contract             0
PaperlessBilling     0
PaymentMethod        0
MonthlyCharges       0
TotalCharges        11
Churn                0
dtype: int64
customer_id          object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object


In [12]:
df.dropna(inplace=True)

In [13]:
df.isnull().sum()

customer_id         0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

In [14]:
dim_customer['customer_id'].duplicated().sum()

np.int64(0)

In [16]:
dim_customer.to_sql("dim_customer", engine, if_exists="append", index=False)
dim_services.to_sql("dim_services", engine, if_exists="append", index=False)
dim_account.to_sql("dim_account", engine, if_exists="append", index=False)
fact_customer_churn.to_sql("fact_customer_churn", engine, if_exists="append", index=False)

7043